# Bronze Layer - Raw Data Ingestion

This notebook is responsible for ingesting the Olist e-commerce dataset
into the Bronze layer using Apache Spark and Delta Lake.

## What we do

- Read raw CSV files from the Unity Catalog Volume.
- Use Spark to create DataFrames from the source data.
- Inspect the inferred schemas.
- Write the raw datasets as Delta tables in the Bronze layer.

## Why Delta Lake?

The source data is provided as CSV files, but CSV is mainly a raw file format.
For the data platform, we store the ingested data as Delta tables.

Delta Lake provides features that are important for reliable data pipelines:

- ACID transactions
- Schema enforcement
- Schema evolution
- Time travel
- Reliable updates and `MERGE` operations

At this stage, we are not applying business transformations.
The Bronze layer keeps the source data as close to its original form as possible.

## Architecture

```text
Olist CSV Files
      ↓
Spark DataFrames
      ↓
Bronze Delta Tables
```

The Silver layer will be responsible for cleaning, validation, joins and other transformations.

In [0]:
print("Databricks E-Commerce Data Platform")
print("Spark version:", spark.version)

In [0]:
%fs ls /Volumes/workspace/default/olist_data/

In [0]:
DATA_PATH = "/Volumes/workspace/default/olist_data"

ORDERS_PATH = f"{DATA_PATH}/olist_orders_dataset.csv"
CUSTOMERS_PATH = f"{DATA_PATH}/olist_customers_dataset.csv"
PRODUCTS_PATH = f"{DATA_PATH}/olist_products_dataset.csv"
ORDER_ITEMS_PATH = f"{DATA_PATH}/olist_order_items_dataset.csv"
PAYMENTS_PATH = f"{DATA_PATH}/olist_order_payments_dataset.csv"

In [0]:
orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ORDERS_PATH)
)

orders_df.printSchema()

In [0]:
display(orders_df)

In [0]:
BRONZE_PATH = "/Volumes/workspace/default/olist_data/bronze/orders"

(
    orders_df.write
    .format("delta")
    .mode("overwrite")
    .save(BRONZE_PATH)
)

In [0]:
bronze_orders_df = spark.read.format("delta").load(BRONZE_PATH)

display(bronze_orders_df)

In [0]:
def load_to_bronze(file_name, table_name):
    source_path = f"{DATA_PATH}/{file_name}"
    bronze_path = f"{DATA_PATH}/bronze/{table_name}"

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_path)
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(bronze_path)
    )

    print(f"{table_name} -> Bronze completed")

In [0]:
load_to_bronze(
    "olist_customers_dataset.csv",
    "customers"
)

load_to_bronze(
    "olist_products_dataset.csv",
    "products"
)

load_to_bronze(
    "olist_order_items_dataset.csv",
    "order_items"
)

load_to_bronze(
    "olist_order_payments_dataset.csv",
    "payments"
)